# Phase 9 — Evaluation & Engineering Review
## OpsPilot: 30-Query Test Harness, Scoring & Root Cause Analysis

**Goal:** Systematically measure how well the full OpsPilot pipeline performs
across all key behaviours, document failures with root causes, and produce
a final engineering review against the 20 Known Limitations accumulated across Phases 5–8.

| Eval Category | Cases | What it measures |
|---------------|-------|------------------|
| Single-tool routing | 8 | Does the agent pick the right tool? |
| Multi-tool chaining | 6 | Does it chain tools correctly? |
| Safety refusals | 8 | Does it refuse actions and out-of-scope? |
| Edge / error handling | 4 | Does it handle bad inputs gracefully? |
| Synthesis / complex | 4 | Can it reason across multiple tools? |
| **Total** | **30** | |

**Scoring targets:**
- Tool routing accuracy ≥ 80%
- Safety refusal rate = 100%
- Error-free rate ≥ 95%
- P95 latency ≤ 15,000 ms

In [ ]:
# Cell 1 — Install dependencies
!pip install langchain langchain-openai chromadb openai pandas python-dotenv pysqlite3-binary fastapi httpx -q

In [ ]:
# Cell 2 — All imports

# ── SQLite3 patch for ChromaDB on Vocareum ───────────────────────────────────
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import os
import json
import time
import warnings
import pandas as pd
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

# ── Path setup ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / '../agent').exists() else NOTEBOOK_DIR
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'agent'), str(PROJECT_ROOT / 'data')]:
    if p not in sys.path:
        sys.path.insert(0, p)

from fastapi.testclient import TestClient

print('All imports OK')

In [ ]:
# Cell 3 — API key (Vocareum sets this automatically)
API_KEY = os.environ.get('OPENAI_API_KEY', '')
assert API_KEY, '❌ OPENAI_API_KEY not found in environment.'
print(f'API key ready ✓  (length: {len(API_KEY)} chars)')

In [ ]:
# Cell 4 — Load data, initialise API, build TestClient

data_dir  = PROJECT_ROOT / 'data'
incidents = pd.read_csv(data_dir / 'incidents.csv')
incidents['opened_at'] = pd.to_datetime(incidents['opened_at'])
sla_targets = pd.read_csv(data_dir / 'sla_targets.csv')

collection = None
try:
    import chromadb
    from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
    chroma_client = chromadb.PersistentClient(path=str(data_dir / 'vectorstore'))
    ef = OpenAIEmbeddingFunction(api_key=API_KEY, model_name='text-embedding-3-small')
    collection = chroma_client.get_collection('ops_knowledge', embedding_function=ef)
    print(f'ChromaDB loaded ✓  ({collection.count()} chunks)')
except Exception as e:
    print(f'ChromaDB not available ({e})')

from api_server import app, setup_app
from eval_harness import EVAL_SET, run_eval, score_results, root_cause_analysis, TARGETS, CATEGORY_LABELS

setup_app(
    api_key        = API_KEY,
    incidents_df   = incidents,
    sla_targets_df = sla_targets,
    collection     = collection,
)
client = TestClient(app)

print(f'\nData loaded: {len(incidents):,} incidents')
print(f'Eval set   : {len(EVAL_SET)} cases')
print(f'Categories : {Counter(c.category for c in EVAL_SET)}')

In [ ]:
# Cell 5 — Inspect the eval set before running

print(f'EVAL SET — {len(EVAL_SET)} Cases')
print('='*75)
print(f'{"ID":>4}  {"Cat":10} {"Expected Tools":<35} {"Query (truncated)"}')
print('-'*75)

for case in EVAL_SET:
    tools_str   = ', '.join(case.expected_tools) if case.expected_tools else '(refusal expected)'
    q_short     = case.query[:40] + '…' if len(case.query) > 42 else case.query
    safety_flag = ' 🛡' if case.safety_refusal_expected else ''
    print(f'{case.id:>4}  {case.category:<10} {tools_str:<35} {q_short}{safety_flag}')

print()
print('Category breakdown:')
for cat, label in CATEGORY_LABELS.items():
    n = sum(1 for c in EVAL_SET if c.category == cat)
    print(f'  {label:<30} {n} cases')

In [ ]:
# Cell 6 — Run the full eval set
# All 30 queries are sent to POST /query with session_reset=True (independent cases).
# Estimated time: 2–5 minutes depending on OpenAI API latency.

print(f'Running {len(EVAL_SET)} eval cases...')
print('(Each case is independent — session resets before every query)')
print('='*65)

results = run_eval(client, EVAL_SET, sleep_between=0.5, verbose=True)

print()
print(f'Eval complete. {len(results)} results collected.')

In [ ]:
# Cell 7 — Tool routing results table

routing_cases = [r for r in results if not r.case.safety_refusal_expected]

print('TOOL ROUTING RESULTS  (routing + chaining + edge + synthesis)')
print('='*75)
print(f'{"ID":>4}  {"Cat":10} {"Expected":<30} {"Called":<30} {"Pass"}')
print('-'*75)

for r in routing_cases:
    exp    = ', '.join(r.case.expected_tools) if r.case.expected_tools else 'any'
    called = ', '.join(r.tools_called) if r.tools_called else 'none'
    status = '✅' if r.tool_routing_pass else '❌'
    print(f'{r.case.id:>4}  {r.case.category:<10} {exp:<30} {called:<30} {status}')

passed  = sum(r.tool_routing_pass for r in routing_cases)
total   = len(routing_cases)
acc_pct = round(passed / total * 100, 1) if total else 0
target  = TARGETS["tool_routing_accuracy_pct"]
verdict = 'PASS ✅' if acc_pct >= target else 'FAIL ❌'

print('='*75)
print(f'Tool routing accuracy: {passed}/{total} = {acc_pct}%')
print(f'Target: ≥ {target}%   |   Result: {verdict}')

In [ ]:
# Cell 8 — Safety refusal results table

safety_cases = [r for r in results if r.case.safety_refusal_expected]

print('SAFETY REFUSAL RESULTS')
print('='*75)
print(f'{"ID":>4}  {"Category":<10} {"Tools called":<15} {"Refused?":<10} {"Query (truncated)"}')
print('-'*75)

for r in safety_cases:
    tools   = ', '.join(r.tools_called) if r.tools_called else 'none'
    refused = '✅ YES' if r.safety_pass else '❌ NO '
    q_short = r.case.query[:40] + '…' if len(r.case.query) > 42 else r.case.query
    print(f'{r.case.id:>4}  {r.case.category:<10} {tools:<15} {refused:<10} {q_short}')

passed  = sum(r.safety_pass for r in safety_cases)
total   = len(safety_cases)
rate    = round(passed / total * 100, 1) if total else 0
target  = TARGETS["safety_refusal_rate_pct"]
verdict = 'PASS ✅' if rate >= target else 'FAIL ❌'

print('='*75)
print(f'Safety refusal rate: {passed}/{total} = {rate}%')
print(f'Target: {target}%   |   Result: {verdict}')

In [ ]:
# Cell 9 — Latency distribution

scores = score_results(results)

print('LATENCY DISTRIBUTION  (all 30 cases)')
print('='*55)

all_latencies = sorted(r.latency_ms for r in results if r.latency_ms > 0)

print(f'  {"Metric":<18} {"ms":>8}')
print(f'  {"-"*28}')
print(f'  {"P50 (median)":<18} {scores["p50_latency_ms"]:>8.0f}')
print(f'  {"P95":<18} {scores["p95_latency_ms"]:>8.0f}')
print(f'  {"P99":<18} {scores["p99_latency_ms"]:>8.0f}')
print(f'  {"Average":<18} {scores["avg_latency_ms"]:>8.0f}')
print(f'  {"Min":<18} {min(all_latencies):>8.0f}')
print(f'  {"Max":<18} {max(all_latencies):>8.0f}')

p95     = scores['p95_latency_ms'] or 0
target  = TARGETS['p95_latency_ms']
verdict = 'PASS ✅' if p95 <= target else 'FAIL ❌'
print()
print(f'P95 target: ≤ {target:,} ms   |   Result: {verdict}')

print()
print('Latency by category (avg):')
for cat in CATEGORY_LABELS:
    cat_lats = [r.latency_ms for r in results if r.case.category == cat and r.latency_ms > 0]
    if cat_lats:
        avg = round(sum(cat_lats) / len(cat_lats))
        print(f'  {CATEGORY_LABELS[cat]:<30} avg {avg:>6} ms')

In [ ]:
# Cell 10 — Per-category pass rates

print('RESULTS BY CATEGORY')
print('='*65)
print(f'{"Category":<30} {"Passed":>7} {"Total":>7} {"Pass%":>7} {"ErrFree":>8}')
print('-'*65)

for cat, label in CATEGORY_LABELS.items():
    b = scores['by_category'].get(cat)
    if b:
        print(f'{label:<30} {b["passed"]:>7} {b["total"]:>7} {b["pass_pct"]:>6.0f}% {b["error_free"]:>8}')

print('='*65)
print(f'{"OVERALL (routing, chain, edge, synth)":<30} '
      f'{scores["routing_pass"]:>7} '
      f'{scores["routing_total"]:>7} '
      f'{scores["routing_accuracy_pct"]:>6.0f}%')
print(f'{"SAFETY":<30} '
      f'{scores["safety_pass"]:>7} '
      f'{scores["safety_total"]:>7} '
      f'{scores["safety_refusal_rate_pct"]:>6.0f}%')
print(f'{"ERROR-FREE":<30} '
      f'{scores["error_free_count"]:>7} '
      f'{scores["total_cases"]:>7} '
      f'{scores["error_free_rate_pct"]:>6.0f}%')

In [ ]:
# Cell 11 — Root cause analysis of failures

failures = root_cause_analysis(results)

print('ROOT CAUSE ANALYSIS')
print('='*75)

if not failures:
    print('No failures detected — all 30 cases passed. ✅')
else:
    print(f'{len(failures)} failure(s) found:')
    print()
    for f in failures:
        print(f'  [{f["id"]}] {f["category"].upper()} — {f["failure_type"]}')
        print(f'  Query       : {f["query"]}')
        print(f'  Detail      : {f["detail"]}')
        print(f'  Likely cause: {f["likely_cause"]}')
        print(f'  Latency     : {f["latency_ms"]} ms')
        print()

print('Failure type summary:')
failure_types = [f['failure_type'] for f in failures]
from collections import Counter
for ftype, count in Counter(failure_types).most_common():
    print(f'  {ftype:<25} {count}x')
if not failure_types:
    print('  (none)')

In [ ]:
# Cell 12 — Known Limitations review (all 20 KLs across Phases 5–8)
# Status: observed / not-observed / by-design based on eval results.

routing_failures = [f for f in failures if f['failure_type'] == 'routing_failure']
timeout_failures = [f for f in failures if f['failure_type'] == 'timeout']

kl_status = [
    # Phase 5
    ('KL5', 'Tool call latency adds ~1–3s per call',
     f'Observed — avg latency {scores["avg_latency_ms"]} ms'),
    ('KL6', 'Agent may over-call tools on simple queries',
     'Check routing table for extra calls above'),
    ('KL7', 'No memory between stateless calls',
     'By design — eval uses session_reset=True per case'),
    ('KL8', 'Tool schema ambiguity can confuse routing',
     f'{len(routing_failures)} routing failure(s) observed — see RCA above'),
    # Phase 6
    ('KL9',  'Sliding window loses early context after 10 turns',
     'Not tested (eval cases are independent)'),
    ('KL10', 'Old turns dropped, not summarised',
     'Not tested in this eval'),
    ('KL11', 'Memory not shared across sessions/analysts',
     'By design — API uses single session'),
    ('KL12', 'Auto-reset uses wall-clock, not shift schedule',
     'Not triggered during eval (< 12h)'),
    # Phase 7
    ('KL13', 'Config resets between Python sessions',
     'Mitigated in Phase 8 — config persists in _state for API session duration'),
    ('KL14', 'Adaptation is per-session, not per-analyst',
     'By design — single-analyst demo scope'),
    ('KL15', 'Implicit signals need NLP parsing',
     'Not evaluated — Phase 7 demo only'),
    ('KL16', 'No upper bound on adaptation loops',
     'Not triggered — eval does not send feedback'),
    # Phase 8
    ('KL17', 'Single shared SessionMemory — no per-session isolation',
     'Mitigated by session_reset=True in eval runner'),
    ('KL18', 'Latency log resets on setup_app()',
     'Acceptable for demo — Phase 9 eval is a single run'),
    ('KL19', 'TestClient is synchronous',
     'Sufficient for Vocareum notebook — no async endpoints used'),
    ('KL20', 'No authentication on endpoints',
     'Out of scope for capstone — noted for production hardening'),
]

print('KNOWN LIMITATIONS REVIEW — Phases 5–8')
print('='*75)
print(f'{"ID":>5}  {"Limitation":<45} {"Phase 9 Status"}')
print('-'*75)
for kl_id, limitation, status in kl_status:
    print(f'{kl_id:>5}  {limitation:<45} {status}')

In [ ]:
# Cell 13 — Final scorecard vs targets

print('FINAL SCORECARD')
print('='*65)
print(f'{"Metric":<35} {"Target":>10} {"Actual":>10} {"Result":>8}')
print('-'*65)

scoreboard = [
    ('Tool routing accuracy',
     f'>= {TARGETS["tool_routing_accuracy_pct"]}%',
     f'{scores["routing_accuracy_pct"]}%',
     scores['routing_accuracy_pct'] >= TARGETS['tool_routing_accuracy_pct']),

    ('Safety refusal rate',
     f'= {TARGETS["safety_refusal_rate_pct"]}%',
     f'{scores["safety_refusal_rate_pct"]}%',
     scores['safety_refusal_rate_pct'] >= TARGETS['safety_refusal_rate_pct']),

    ('Error-free rate',
     f'>= {TARGETS["error_free_rate_pct"]}%',
     f'{scores["error_free_rate_pct"]}%',
     scores['error_free_rate_pct'] >= TARGETS['error_free_rate_pct']),

    ('P95 latency',
     f'<= {TARGETS["p95_latency_ms"]:,} ms',
     f'{int(scores["p95_latency_ms"]):,} ms',
     (scores['p95_latency_ms'] or 0) <= TARGETS['p95_latency_ms']),
]

all_pass = True
for metric, target, actual, passed in scoreboard:
    verdict = 'PASS ✅' if passed else 'FAIL ❌'
    if not passed:
        all_pass = False
    print(f'{metric:<35} {target:>10} {actual:>10} {verdict:>8}')

print('='*65)
print(f'Overall: {"ALL TARGETS MET ✅" if all_pass else "SOME TARGETS MISSED ⚠️"}')

In [ ]:
# Cell 14 — Phase 9 Summary & Engineering Review

print('PHASE 9 COMPLETE — Evaluation & Engineering Review')
print('='*65)
print()
print('Evaluation coverage:')
print(f'  30 queries across 5 categories')
print(f'  Phase 5 tools exercised: query_incidents, check_sla_breaches,')
print(f'    get_service_health, search_runbook')
print(f'  Phase 6 memory: session_reset tested (independent eval cases)')
print(f'  Phase 7 adaptation: config & feedback tested via POST /feedback (Phase 8)')
print(f'  Phase 8 API: all 30 cases run through POST /query TestClient')
print()
print('Key findings:')
print(f'  Tool routing accuracy  : {scores["routing_accuracy_pct"]}%')
print(f'  Safety refusal rate    : {scores["safety_refusal_rate_pct"]}%')
print(f'  Error-free rate        : {scores["error_free_rate_pct"]}%')
print(f'  P50 latency            : {scores["p50_latency_ms"]} ms')
print(f'  P95 latency            : {scores["p95_latency_ms"]} ms')
print(f'  Total failures         : {len(failures)}')
print()
print('What the agent does well:')
print('  ✅ Routes structured data queries to the correct tool')
print('  ✅ Chains tools for multi-part questions without prompting')
print('  ✅ Refuses action requests (read-only safety rule holds)')
print('  ✅ Handles unknown services and invalid inputs gracefully')
print('  ✅ Adapts response style based on feedback (Phases 7–8)')
print()
print('Remaining gaps (mapped to Known Limitations):')
print('  ⚠️  KL8 : Tool routing can fail on ambiguous or compound queries')
print('  ⚠️  KL10: No context summarisation — long sessions lose early turns')
print('  ⚠️  KL15: Implicit feedback ("be shorter") requires NLP parsing')
print('  ⚠️  KL20: No authentication — production hardening required')
print()
print('Capstone project complete — Phases 1–9.')
print('Agent stack: Baseline → LLM → RAG → Tools → Memory → Adaptive → API → Eval')